## Runtime Diagnostics
Run this first after kernel restart to confirm the notebook is using the project environment and to get the active Spark UI URL.

In [1]:
import os
import sys
from pathlib import Path

# Resolve project root regardless of whether notebook cwd is project root or notebooks/.
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent

# Force local Java/Hadoop runtime used by this project (avoids Java 24 incompatibility).
os.environ["JAVA_HOME"] = str(project_root / "jdk-17.0.2")
os.environ["HADOOP_HOME"] = str(project_root / "hadoop")
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["PATH"] = (
    str(project_root / "jdk-17.0.2" / "bin")
    + os.pathsep
    + str(project_root / "hadoop" / "bin")
    + os.pathsep
    + os.environ.get("PATH", "")
)

import pyspark
from pyspark.sql import SparkSession

print("Python executable:", sys.executable)
print("PySpark version:", pyspark.__version__)
print("JAVA_HOME:", os.environ["JAVA_HOME"])

spark = (
    SparkSession.builder.appName("NotebookSession")
    .master("local[*]")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.python.use.daemon", "false")
    .config("spark.driver.extraJavaOptions", "-Duser.name=notebook")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark UI URL:", spark.sparkContext.uiWebUrl)

Python executable: D:\python\bigdata\.venv\Scripts\python.exe
PySpark version: 3.5.0
JAVA_HOME: D:\python\bigdata\jdk-17.0.2
Spark UI URL: http://127.0.0.1:4040


# PySpark Big Data Learning
## GPS & Traffic Data Analysis from UK Government Sources

This notebook demonstrates PySpark fundamentals using real-world UK traffic data.

## 1. Initialize Spark Session

In [2]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, avg, count, max as spark_max, min as spark_min

# Make src import work in local notebook execution.
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Reuse existing Spark session if already created by diagnostics cell.
spark = SparkSession.getActiveSession() or (
    SparkSession.builder
    .appName("BigDataLearning")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.python.use.daemon", "false")
    .config("spark.driver.extraJavaOptions", "-Duser.name=notebook")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

Spark Version: 3.5.0
Master: local[*]


## 2. Load GPS Data

In [3]:
spark.sparkContext.uiWebUrl

'http://127.0.0.1:4040'

In [4]:
from pathlib import Path
from datetime import datetime
from src.data_fetcher import generate_sample_gps_data

# Safe conversion path on Windows: pandas -> CSV -> Spark read (JVM path)
def pandas_to_spark_via_csv(pdf, name_prefix="data"):
    cwd = Path.cwd()
    root = cwd if (cwd / "src").exists() else cwd.parent
    tmp_dir = root / "data" / "_tmp_notebook"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    csv_path = tmp_dir / f"{name_prefix}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.csv"
    pdf.to_csv(csv_path, index=False)
    return spark.read.option("header", True).option("inferSchema", True).csv(str(csv_path))

# Generate and load sample GPS data
pdf = generate_sample_gps_data(10000)
gps_df = pandas_to_spark_via_csv(pdf, "gps")

print(f"Total records: {gps_df.count()}")
print("\nDataFrame Schema:")
gps_df.printSchema()

Total records: 10000

DataFrame Schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- speed_kmh: double (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- location: string (nullable = true)



In [5]:
# Replacement for the failing pandas -> Spark conversion cell
from pathlib import Path
from datetime import datetime
from src.data_fetcher import generate_sample_gps_data

# Safe conversion path on Windows: pandas -> CSV -> Spark read
# This avoids Python worker crashes seen with spark.createDataFrame(pandas_df).
def pandas_to_spark_via_csv(pdf, name_prefix="data"):
    cwd = Path.cwd()
    root = cwd if (cwd / "src").exists() else cwd.parent
    tmp_dir = root / "data" / "_tmp_notebook"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    csv_path = tmp_dir / f"{name_prefix}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.csv"
    pdf.to_csv(csv_path, index=False)
    return spark.read.option("header", True).option("inferSchema", True).csv(str(csv_path))

pdf = generate_sample_gps_data(10000)
gps_df = pandas_to_spark_via_csv(pdf, "gps")

print(f"Total records: {gps_df.count()}")
print("\nDataFrame Schema:")
gps_df.printSchema()

Total records: 10000

DataFrame Schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- speed_kmh: double (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- location: string (nullable = true)



## 3. Explore Data

In [6]:
# Show sample records
print("Sample GPS Data:")
gps_df.show(10)



Sample GPS Data:
+--------------------+------------------+--------------------+------------------+----------+----------+
|           timestamp|          latitude|           longitude|         speed_kmh|vehicle_id|  location|
+--------------------+------------------+--------------------+------------------+----------+----------+
|2026-04-12 11:41:...| 51.56176411223351|-0.08481961421174783|15.152803949991958|  VEH_1596|Birmingham|
|2026-04-12 11:41:...| 51.62064070531092| -0.1951579112144655|50.140816281743994|  VEH_3732|     Leeds|
|2026-04-12 11:41:...| 51.67867351008121| 0.14503510281435372|  73.9838217700486|  VEH_3490|    London|
|2026-04-12 11:41:...| 51.35667025843675|-0.06181280935241373| 55.32928534693592|  VEH_1708|    London|
|2026-04-12 11:41:...|51.359619845870384| -0.2680435301260092|28.062125516292724|  VEH_2703|Birmingham|
|2026-04-12 11:41:...|51.469669107553564| 0.12826122715234622| 62.20194767558141|  VEH_9311|Birmingham|
|2026-04-12 11:41:...| 51.40500621839447| -0.37

In [7]:
# Show statistics
print("\nData Statistics:")
gps_df.describe(['speed_kmh', 'latitude', 'longitude']).show()


Data Statistics:
+-------+--------------------+-------------------+--------------------+
|summary|           speed_kmh|           latitude|           longitude|
+-------+--------------------+-------------------+--------------------+
|  count|               10000|              10000|               10000|
|   mean|  39.777101315970384|  51.50089904184338|-0.09997257075742223|
| stddev|  23.144611027297174|0.11531186369195875| 0.17310303217656908|
|    min|0.011403636326656752|  51.30000014510331|-0.39991352307810446|
|    max|   79.98570491090653|  51.69995432666494|  0.1999385037613789|
+-------+--------------------+-------------------+--------------------+



## 4. Basic Transformations

In [8]:
# Filter: Get high-speed events (> 70 km/h)
high_speed = gps_df.filter(col('speed_kmh') > 70)
print(f"High-speed events (>70 km/h): {high_speed.count()}")
print("\nSample high-speed records:")
high_speed.show(10)

High-speed events (>70 km/h): 1258

Sample high-speed records:
+--------------------+------------------+--------------------+-----------------+----------+----------+
|           timestamp|          latitude|           longitude|        speed_kmh|vehicle_id|  location|
+--------------------+------------------+--------------------+-----------------+----------+----------+
|2026-04-12 11:41:...| 51.67867351008121| 0.14503510281435372| 73.9838217700486|  VEH_3490|    London|
|2026-04-12 11:41:...| 51.53973979351331|-0.09832855161771303|79.87464036830875|  VEH_9086|Birmingham|
|2026-04-12 11:42:...| 51.65464003171837|-0.32083000925265803|77.24117901494685|  VEH_3708|Manchester|
|2026-04-12 11:42:...| 51.34059595499662|  0.0611205850559885|71.19585777120119|  VEH_2436|     Leeds|
|2026-04-12 11:42:...| 51.38259680936141| 0.17149598289020962|75.39260589015548|  VEH_5328|    London|
|2026-04-12 11:42:...| 51.50795609400733|-0.06219320978684678|76.96648510959426|  VEH_7825|     Leeds|
|2026-04-1

In [9]:
# Select specific columns
subset = gps_df.select('timestamp', 'vehicle_id', 'location', 'speed_kmh')
print("Selected columns:")
subset.show(10)

Selected columns:
+--------------------+----------+----------+------------------+
|           timestamp|vehicle_id|  location|         speed_kmh|
+--------------------+----------+----------+------------------+
|2026-04-12 11:41:...|  VEH_1596|Birmingham|15.152803949991958|
|2026-04-12 11:41:...|  VEH_3732|     Leeds|50.140816281743994|
|2026-04-12 11:41:...|  VEH_3490|    London|  73.9838217700486|
|2026-04-12 11:41:...|  VEH_1708|    London| 55.32928534693592|
|2026-04-12 11:41:...|  VEH_2703|Birmingham|28.062125516292724|
|2026-04-12 11:41:...|  VEH_9311|Birmingham| 62.20194767558141|
|2026-04-12 11:41:...|  VEH_1904|Manchester| 49.67416114039794|
|2026-04-12 11:41:...|  VEH_4404|    London|42.547523341145435|
|2026-04-12 11:41:...|  VEH_5061|Manchester|49.636504620474696|
|2026-04-12 11:41:...|  VEH_9086|Birmingham| 79.87464036830875|
+--------------------+----------+----------+------------------+
only showing top 10 rows



## 5. Aggregations & Grouping

In [10]:
# Average speed by location
print("Average Speed by Location:")
gps_df.groupBy('location') \
    .agg({
        'speed_kmh': 'avg',
        'vehicle_id': 'count'
    }) \
    .withColumnRenamed('avg(speed_kmh)', 'avg_speed') \
    .withColumnRenamed('count(vehicle_id)', 'vehicle_count') \
    .sort('avg_speed', ascending=False) \
    .show()

Average Speed by Location:
+----------+-----------------+-------------+
|  location|        avg_speed|vehicle_count|
+----------+-----------------+-------------+
|    London| 40.0438431939108|         2494|
|Birmingham|39.93914323353082|         2480|
|     Leeds|39.91653064943384|         2531|
|Manchester|39.20795749146978|         2495|
+----------+-----------------+-------------+



In [11]:
# Speed statistics by location
print("Speed Statistics by Location:")
gps_df.groupBy('location').agg(
    avg('speed_kmh').alias('avg_speed'),
    spark_min('speed_kmh').alias('min_speed'),
    spark_max('speed_kmh').alias('max_speed'),
    count('speed_kmh').alias('count')
).show()

Speed Statistics by Location:
+----------+-----------------+--------------------+-----------------+-----+
|  location|        avg_speed|           min_speed|        max_speed|count|
+----------+-----------------+--------------------+-----------------+-----+
|Manchester|39.20795749146978|  0.1050259201314585|79.98570491090653| 2495|
|    London| 40.0438431939108| 0.18360736728626925|79.98270105699488| 2494|
|Birmingham|39.93914323353082|0.011403636326656752|79.91346982053052| 2480|
|     Leeds|39.91653064943384| 0.01346660166912983|79.97684000810042| 2531|
+----------+-----------------+--------------------+-----------------+-----+



## 6. Complex Queries with SQL

In [12]:
# Register as temporary table
gps_df.createOrReplaceTempView("gps_data")

# Run SQL queries
result = spark.sql("""
SELECT 
    location,
    COUNT(*) as total_records,
    ROUND(AVG(speed_kmh), 2) as avg_speed,
    MAX(speed_kmh) as max_speed
FROM gps_data
GROUP BY location
ORDER BY avg_speed DESC
""")

print("Speed Analysis by Location (SQL):")
result.show()

Speed Analysis by Location (SQL):
+----------+-------------+---------+-----------------+
|  location|total_records|avg_speed|        max_speed|
+----------+-------------+---------+-----------------+
|    London|         2494|    40.04|79.98270105699488|
|Birmingham|         2480|    39.94|79.91346982053052|
|     Leeds|         2531|    39.92|79.97684000810042|
|Manchester|         2495|    39.21|79.98570491090653|
+----------+-------------+---------+-----------------+



## 7. Performance Analysis

In [13]:
# Find vehicles with high average speed
print("Top 10 Vehicles by Average Speed:")
spark.sql("""
SELECT 
    vehicle_id,
    location,
    ROUND(AVG(speed_kmh), 2) as avg_speed,
    COUNT(*) as num_records
FROM gps_data
GROUP BY vehicle_id, location
ORDER BY avg_speed DESC
LIMIT 10
""").show()

Top 10 Vehicles by Average Speed:
+----------+----------+---------+-----------+
|vehicle_id|  location|avg_speed|num_records|
+----------+----------+---------+-----------+
|  VEH_5798|Manchester|    79.99|          1|
|  VEH_4695|Manchester|    79.98|          1|
|  VEH_5995|     Leeds|    79.98|          1|
|  VEH_6666|    London|    79.98|          1|
|  VEH_3990|    London|    79.98|          1|
|  VEH_7741|    London|    79.98|          1|
|  VEH_5431|    London|    79.96|          1|
|  VEH_2512|    London|    79.92|          1|
|  VEH_2417|Birmingham|    79.91|          1|
|  VEH_9427|     Leeds|     79.9|          1|
+----------+----------+---------+-----------+



## 8. Load UK Government Data

In [14]:
from src.data_fetcher import UKGovDataFetcher

# Ensure helper exists even if cells were run out of order
if "pandas_to_spark_via_csv" not in globals():
    from pathlib import Path
    from datetime import datetime
    def pandas_to_spark_via_csv(pdf, name_prefix="data"):
        cwd = Path.cwd()
        root = cwd if (cwd / "src").exists() else cwd.parent
        tmp_dir = root / "data" / "_tmp_notebook"
        tmp_dir.mkdir(parents=True, exist_ok=True)
        csv_path = tmp_dir / f"{name_prefix}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.csv"
        pdf.to_csv(csv_path, index=False)
        return spark.read.option("header", True).option("inferSchema", True).csv(str(csv_path))

fetcher = UKGovDataFetcher()

print("Fetching TfL Traffic Data...")
traffic_pdf = fetcher.fetch_tfl_traffic_data()

if not traffic_pdf.empty:
    traffic_df = pandas_to_spark_via_csv(traffic_pdf, "traffic")
    print(f"\nTraffic Incidents: {traffic_df.count()}")
    traffic_df.show(20)
else:
    print("No traffic data available at this time")

Fetching TfL Traffic Data...

Traffic Incidents: 74
+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|      disruption_id|            category|            severity|            location|            comments|       last_modified|           timestamp|         data_source|       endpoint_used|
+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|        TIMS-206772|               Works|            Moderate|[A12] EASTERN AVE...|Gallows Corner Fl...|                NULL|                NULL|                NULL|                NULL|
|Between 10-13 April| Gallows Corner r...| to facilitate re...| A12 Colchester R...|2026-04-12T03:34:31Z|2026-04-12 11:42:...|  TfL_RoadDisruption|https://api.tfl.g...|                NULL

In [15]:
print("\nFetching TfL Line Status...")
line_pdf = fetcher.fetch_tfl_line_status()

if not line_pdf.empty:
    line_df = pandas_to_spark_via_csv(line_pdf, "line_status")
    print(f"\nTube Line Status: {line_df.count()} lines")
    line_df.show(20)
else:
    print("No line status data available at this time")


Fetching TfL Line Status...

Tube Line Status: 11 lines
+----------------+------------------+---------------+--------------------+--------------+
|         line_id|         line_name|         status|           timestamp|   data_source|
+----------------+------------------+---------------+--------------------+--------------+
|        bakerloo|          Bakerloo|   Good Service|2026-04-12 11:42:...|TfL_LineStatus|
|         central|           Central|   Good Service|2026-04-12 11:42:...|TfL_LineStatus|
|          circle|            Circle|   Part Closure|2026-04-12 11:42:...|TfL_LineStatus|
|        district|          District|   Part Closure|2026-04-12 11:42:...|TfL_LineStatus|
|hammersmith-city|Hammersmith & City|   Good Service|2026-04-12 11:42:...|TfL_LineStatus|
|         jubilee|           Jubilee|   Good Service|2026-04-12 11:42:...|TfL_LineStatus|
|    metropolitan|      Metropolitan|   Good Service|2026-04-12 11:42:...|TfL_LineStatus|
|        northern|          Northern|   Goo

## 9. Demonstrate Spark Execution Plan

In [16]:
# Show execution plan
query = gps_df.groupBy('location').agg(avg('speed_kmh')).sort('location')

print("Logical Plan:")
print(query.explain())

print("\nResults:")
query.show()

Logical Plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [location#62 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(location#62 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=581]
      +- HashAggregate(keys=[location#62], functions=[avg(speed_kmh#60)])
         +- Exchange hashpartitioning(location#62, 200), ENSURE_REQUIREMENTS, [plan_id=578]
            +- HashAggregate(keys=[location#62], functions=[partial_avg(speed_kmh#60)])
               +- FileScan csv [speed_kmh#60,location#62] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/D:/python/bigdata/data/_tmp_notebook/gps_20260412_114147_722575...., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<speed_kmh:double,location:string>


None

Results:
+----------+-----------------+
|  location|   avg(speed_kmh)|
+----------+-----------------+
|Birmingham|39.93914323353082|
|     Leeds|39.91653064943384|
|    London| 40.0438431939108|
|Manchester|3

## 10. Summary

In [17]:
print("="*60)
print("SUMMARY - PySpark Big Data Learning")
print("="*60)

print(f"\n✓ Spark Session: Active")
print(f"✓ Total GPS Records Processed: {gps_df.count():,}")
print(f"✓ Locations Covered: {gps_df.select('location').distinct().count()}")
print(f"✓ Average Speed Overall: {gps_df.agg(avg('speed_kmh')).collect()[0][0]:.2f} km/h")
print(f"✓ Data Source: UK Gov (TfL) + Synthetic GPS")
print("\nNext Steps:")
print("1. Try modifying the queries above")
print("2. Increase data volume (change 10000 to larger number)")
print("3. Add windowing functions for time-series analysis")
print("4. Export results to Parquet format")
print("5. Monitor the Spark UI at http://localhost:4040")

SUMMARY - PySpark Big Data Learning

✓ Spark Session: Active
✓ Total GPS Records Processed: 10,000
✓ Locations Covered: 4
✓ Average Speed Overall: 39.78 km/h
✓ Data Source: UK Gov (TfL) + Synthetic GPS

Next Steps:
1. Try modifying the queries above
2. Increase data volume (change 10000 to larger number)
3. Add windowing functions for time-series analysis
4. Export results to Parquet format
5. Monitor the Spark UI at http://localhost:4040
